# 特徴量エンジニアリング

モデルの種類を変える前に、入力を変えるだけで性能が大きく変わることがあります。特徴量エンジニアリングは、元データを「モデルが読み取りやすい形」に翻訳する作業です。

このノートでは、不動産価格のような疑似データを題材に、型と欠損の確認、分布変換、ドメイン知識による特徴追加、交互作用、木モデルとの比較までを順に見ます。

## まずは、モデルより前に表現を整える

特徴量設計の重要さは、複雑なモデルを使う前に入力の質を上げられるところにあります。同じ回帰モデルでも、距離の分布を変換したり、面積と部屋数の関係を新しい列にしたりするだけで、見え方がかなり変わります。


このノートでは、最初にデータの列と欠損を観察し、次に歪んだ分布を変換し、ベースラインを作り、その上で意味のある特徴を追加して比較します。最後に、線形モデルと木モデルで効き方がどう違うかも見ます。

つまり、特徴量エンジニアリングを「思いつきの列追加」ではなく、比較可能な改善作業として扱う構成です。


## このノートで使う基本語彙

- 欠損値: 値が入っていないセル
- 標準化: 列のスケールをそろえる変換
- One-Hot エンコード: カテゴリを 0/1 列に分ける変換
- 交互作用: 2 つの特徴量を掛け合わせて作る項

重要なのは、これらを個別技法として覚えることより、「どの変換がどんなモデルを助けるのか」を比較しながら掴むことです。


## 変換の前後で、何が変わったかを見る

このノートでは、常にベースラインとの比較を意識します。型や欠損を確認し、変換前後の分布を見て、評価指標の差を比べる。この順番を守ると、改善の理由を言葉で説明しやすくなります。


## 先に構えたい失敗

特徴量を増やすと、訓練データには合わせやすくなります。その代わり、意味の薄い列やテスト情報を混ぜると、簡単にリークや過学習が起こります。

`ColumnTransformer` や `Pipeline` を使うのは、変換を整理するためだけでなく、その事故を防ぐためでもあります。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures, FunctionTransformer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor

sns.set_theme(style="whitegrid", context="notebook")


## 1. まずは元データの癖を読む

特徴量設計の出発点は、列の意味と型を把握することです。どの列が数値で、どの列がカテゴリで、どこに欠損があるかが見えていないと、変換はたいてい空回りします。

In [ ]:
rng = np.random.default_rng(42)
n = 600

area = rng.normal(65, 18, n).clip(20, 150)
rooms = rng.integers(1, 6, n)
age = rng.integers(0, 40, n)
distance = rng.normal(18, 8, n).clip(1, 45)
station = rng.choice(["A", "B", "C", "D"], size=n, p=[0.30, 0.30, 0.25, 0.15])
floor = rng.integers(1, 16, n)

station_effect = pd.Series(station).map({"A": 180, "B": 120, "C": 60, "D": 20}).to_numpy()
noise = rng.normal(0, 35, n)
price = 45 + 4.2 * area + 16 * rooms - 2.3 * age - 3.8 * distance + station_effect + 2.8 * floor + noise

df = pd.DataFrame({
    "area": area,
    "rooms": rooms,
    "age": age,
    "distance_to_station": distance,
    "station": station,
    "floor": floor,
    "price": price,
})

# 欠損を意図的に作る
missing_idx_num = rng.choice(df.index, size=35, replace=False)
missing_idx_cat = rng.choice(df.index, size=20, replace=False)
df.loc[missing_idx_num, "distance_to_station"] = np.nan
df.loc[missing_idx_cat, "station"] = np.nan

df.head()


`info` と `isna` を見るのは、あとで変換エラーや思わぬリークを避けるためです。特徴量設計は、派手な新列より先に現状把握から始まります。

In [ ]:
print(df.info())
print("\n欠損数:\n", df.isna().sum())


## 2. 分布をそのまま使うか、変換するか

歪んだ分布は、そのままだと線形モデルで扱いづらいことがあります。ここでは駅距離を例に、`log1p` の前後で見え方がどう変わるかを確かめます。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))

sns.histplot(df["distance_to_station"], bins=30, ax=axes[0], kde=True)
axes[0].set_title("original distance")

sns.histplot(np.log1p(df["distance_to_station"]), bins=30, ax=axes[1], kde=True)
axes[1].set_title("log1p(distance)")

plt.tight_layout()
plt.show()


## 3. 比較のための素朴なベースラインを置く

最初に単純な前処理だけでモデルを作っておくと、あとで加えた特徴の効果を測りやすくなります。特徴量設計では「どれだけ工夫したか」より「何を足したらどれだけ変わったか」を比べることが大切です。

In [ ]:
X = df.drop(columns=["price"])
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

num_cols = ["area", "rooms", "age", "distance_to_station", "floor"]
cat_cols = ["station"]

baseline_preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), cat_cols),
])

baseline_model = Pipeline([
    ("preprocess", baseline_preprocess),
    ("regressor", LinearRegression()),
])

baseline_model.fit(X_train, y_train)
baseline_pred = baseline_model.predict(X_test)

print(f"baseline MAE: {mean_absolute_error(y_test, baseline_pred):.2f}")
print(f"baseline R2 : {r2_score(y_test, baseline_pred):.3f}")


## 4. 意味のある特徴を足してみる

ここで追加するのは、単に列数を増やすための特徴ではありません。`area_per_room` は広さと間取りを同時に見た指標、`is_new` は閾値で効くかもしれない築浅効果、`log_distance` は歪みを抑えた距離表現です。

ドメイン知識とは、こうした「その列を追加する理由を言えること」に近いです。

In [ ]:
def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out["area_per_room"] = out["area"] / np.maximum(out["rooms"], 1)
    out["is_new"] = (out["age"] <= 5).astype(int)
    out["log_distance"] = np.log1p(out["distance_to_station"])
    return out

feature_num_cols = [
    "area", "rooms", "age", "distance_to_station", "floor",
    "area_per_room", "is_new", "log_distance"
]
feature_cat_cols = ["station"]

feature_preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), feature_num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), feature_cat_cols),
])

feature_model = Pipeline([
    ("feature_builder", FunctionTransformer(add_features, validate=False)),
    ("preprocess", feature_preprocess),
    ("regressor", Ridge(alpha=1.0, random_state=42)),
])

feature_model.fit(X_train, y_train)
feature_pred = feature_model.predict(X_test)

print(f"feature MAE: {mean_absolute_error(y_test, feature_pred):.2f}")
print(f"feature R2 : {r2_score(y_test, feature_pred):.3f}")


## 5. 掛け算の項で表現力を増やす

線形モデルでも、交互作用や二乗項を入れると表現力は上がります。ただし、そのぶん列数も一気に増えます。改善と過学習リスクが同時に増えるので、必ず比較しながら使う必要があります。

In [ ]:
poly_num_cols = ["area", "rooms", "age", "distance_to_station", "floor"]

poly_preview_input = X_train[poly_num_cols].fillna(X_train[poly_num_cols].median())
poly_preview = PolynomialFeatures(degree=2, include_bias=False)
poly_preview.fit(poly_preview_input)
poly_names = poly_preview.get_feature_names_out(poly_num_cols)
print("num of polynomial features:", len(poly_names))
print("sample names:", poly_names[:12])

poly_preprocess = ColumnTransformer([
    ("num_poly", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("scaler", StandardScaler()),
    ]), poly_num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), cat_cols),
])

poly_model = Pipeline([
    ("preprocess", poly_preprocess),
    ("regressor", Ridge(alpha=2.0, random_state=42)),
])

poly_model.fit(X_train, y_train)
poly_pred = poly_model.predict(X_test)

print(f"poly MAE: {mean_absolute_error(y_test, poly_pred):.2f}")
print(f"poly R2 : {r2_score(y_test, poly_pred):.3f}")


## 6. 木モデルだと何が変わるか

木ベースモデルは、スケーリングや単純な非線形性に比較的強いので、特徴量設計との相性を見比べる相手として便利です。ここでは、線形モデルで効いた工夫が木モデルでも効くのかを確認します。

In [ ]:
tree_preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ]), feature_num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), feature_cat_cols),
])

tree_model = Pipeline([
    ("feature_builder", FunctionTransformer(add_features, validate=False)),
    ("preprocess", tree_preprocess),
    ("regressor", RandomForestRegressor(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    )),
])

tree_model.fit(X_train, y_train)
tree_pred = tree_model.predict(X_test)

print(f"random forest MAE: {mean_absolute_error(y_test, tree_pred):.2f}")
print(f"random forest R2 : {r2_score(y_test, tree_pred):.3f}")


## 7. いちばん危ないのはリーク

特徴量エンジニアリングで最も危険なのは、テスト側の情報を前処理へ混ぜてしまうことです。欠損補完やスケーリングの fit をどこで行うかまで含めて設計しないと、見かけだけ高いスコアが簡単に作れてしまいます。

In [ ]:
models = {
    "baseline_linear": baseline_model,
    "feature_ridge": feature_model,
    "poly_ridge": poly_model,
    "tree_rf": tree_model,
}

rows = []
for name, model in models.items():
    # すべての前処理（特徴量追加を含む）をパイプライン内で実行
    scores = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring="neg_mean_absolute_error",
        n_jobs=-1,
    )
    rows.append({
        "model": name,
        "cv_mae_mean": -scores.mean(),
        "cv_mae_std": scores.std(),
    })

pd.DataFrame(rows).sort_values("cv_mae_mean")


## まとめ

特徴量設計で大事なのは、思いついた列を増やすことではなく、元データを観察し、ベースラインを作り、意味のある変換だけを追加して比較することです。モデル選択より前に、入力表現の質が性能をかなり左右することをこのノートで掴んでください。